# Total Care Traffic — KITTI Tracking 2단계 실물 감사 및 3단계 시퀀스 분할 (Colab + Google Drive)

- **도메인:** 교통·자율주행 (외부 `Totalcare_autonomous_driving_Yolov5` 확장 트랙)
- **실행 환경:** Google Colab + Google Drive (로컬 디스크에 `14.73 GiB`를 받지 않고 Colab/Drive에서 직접 확보·검증)
- **검증 절차 (Phase 0 -> Phase 1):**
  1. **2단계:** S3 공식 경로에서 `data_tracking_image_2.zip` (`14.73 GiB`) + `data_tracking_label_2.zip` (`2.15 MiB`) 다운로드 → ZIP 무결성(`testzip()`), 이미지 수·해상도·FPS(`10 Hz`), 라벨 프레임 인덱스 1:1 대응(빈 라벨 84프레임 포함) 검증
  2. **3단계:** 21개 학습 시퀀스를 **시퀀스 단위 비겹침 Train / Val / Test**로 분할하고, 각 split 내 목표 클래스 분포 및 `(sequence_id, track_id)` 50프레임(`5초`) 유지 여부 점검
  3. **주의:** 공식 KITTI TrackEval 평가는 `Car`·`Pedestrian` 2종 한정(`Van`/`Person` distractor 무시, `DontCare` crowd ignore)이며, 제3 클래스 포함 평가는 별도 자체 프로토콜로 분리합니다. 라이선스(`CC BY-NC-SA 3.0`)의 가중치·5초 추론 동영상 적용 범위도 별도 검토 대상입니다.


In [ ]:
# 1. Google Drive 마운트 및 경로 설정
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
KITTI_DIR = Path('/content/drive/MyDrive/KITTI_Tracking')
KITTI_DIR.mkdir(parents=True, exist_ok=True)

IMG_ZIP = KITTI_DIR / 'data_tracking_image_2.zip'
LBL_ZIP = KITTI_DIR / 'data_tracking_label_2.zip'
WORK_YOLO_DIR = Path('/content/kitti_yolo')


In [ ]:
# 2. 저장소 동기화 및 필수 패키지 확인
# !git clone https://github.com/Heisnotanimposter/Totalcare_autonomous_driving_Yolov5.git /content/TotalCare
%cd /content/TotalCare
!pip install -q -r requirements.txt


In [ ]:
# 3. KITTI Tracking 공식 S3 다운로드 (이미 Drive에 있으면 건너뜀)
if not LBL_ZIP.exists():
    !wget -c -O "{LBL_ZIP}" https://s3.eu-central-1.amazonaws.com/avg-kitti/data_tracking_label_2.zip
if not IMG_ZIP.exists():
    !wget -c -O "{IMG_ZIP}" https://s3.eu-central-1.amazonaws.com/avg-kitti/data_tracking_image_2.zip

!ls -lh "{KITTI_DIR}" 


In [ ]:
# 4. 2단계 실물 감사: ZIP 무결성, 8,008프레임 1:1 대응, 해상도/채널 실측
!python3 src/kitti_prep.py audit --image-zip "{IMG_ZIP}" --label-zip "{LBL_ZIP}" > "{KITTI_DIR / 'kitti_step2_audit.json'}"
import json
audit = json.loads((KITTI_DIR / 'kitti_step2_audit.json').read_text())
print("Corrupt member:", audit['image_zip_corrupt_member'])
print("Total images:", audit['total_training_images'], "Expected seqmap:", audit['expected_seqmap_frames'])
print("Sampled resolutions:", audit['sampled_resolutions'])


In [ ]:
# 5. 3단계 시퀀스 단위 분할 사전 점검 (예시 분할 — 최종 확정 전 각 split 클래스·tracklet 분포 검증용)
import sys
sys.path.insert(0, '/content/TotalCare/src')
from kitti_prep import load_labels_from_zip, validate_disjoint_sequences
from collections import Counter

labels = load_labels_from_zip(LBL_ZIP)
# 예시: 21개 시퀀스를 시퀀스 단위로 비겹침 분할 (최종 분할은 아래 분포표를 보고 조정)
TRAIN_SEQS = ["0000", "0001", "0003", "0004", "0005", "0006", "0007", "0008", "0009", "0010", "0011", "0014", "0018", "0020"]
VAL_SEQS   = ["0002", "0012", "0015", "0017"]
TEST_SEQS  = ["0013", "0016", "0019"]
validate_disjoint_sequences(TRAIN_SEQS, VAL_SEQS, TEST_SEQS)

for split_name, seqs in [("train", TRAIN_SEQS), ("val", VAL_SEQS), ("test", TEST_SEQS)]:
    c = Counter()
    t = 0
    for s in seqs:
        c.update(labels[s]['class_counts'])
        t += labels[s]['sequence_scoped_tracklets']
    print(f"[{split_name}] seqs={len(seqs)} tracklets={t} | classes={dict(c)}")


In [ ]:
# 6. DontCare/Distractor 마스킹 적용 Sequence-Split YOLO Export
# 목표 클래스는 최종 결정에 맞게 수정하세요 (예: Car Pedestrian Cyclist 또는 Car Pedestrian Van 등)
TARGET_CLASSES = "Car Pedestrian Cyclist"
!python3 src/kitti_prep.py export \
  --image-zip "{IMG_ZIP}" \
  --label-zip "{LBL_ZIP}" \
  --output-dir "{WORK_YOLO_DIR}" \
  --classes {TARGET_CLASSES} \
  --train-seqs {" ".join(TRAIN_SEQS)} \
  --val-seqs {" ".join(VAL_SEQS)} \
  --test-seqs {" ".join(TEST_SEQS)}
